In [1]:
import Pkg
Pkg.update()
Pkg.add("PhysOcean")
Pkg.add("Downloads")
Pkg.add("Makie")
Pkg.add("CairoMakie")
Pkg.add("GeoMakie")

using NCDatasets
using PhysOcean
using Downloads
using DataStructures
using DIVAnd
using Dates
using Statistics
using Random
using Makie, CairoMakie, GeoMakie
#include("../config.jl")

   Updating git-repo `https://github.com/gher-ulg/DIVAndNN.jl`


Updating `/opt/julia/environments/v1.5/Project.toml`
  [1b7aed40] ~ DIVAndNN v0.1.0 `https://github.com/gher-ulg/DIVAndNN.jl#master` ⇒ v0.1.0 `https://github.com/gher-ulg/DIVAndNN.jl#master`
  [f67ccb44] ↑ HDF5 v0.13.5 ⇒ v0.15.5
  [85f8d34a] ↑ NCDatasets v0.10.4 ⇒ v0.11.5
Updating `/opt/julia/environments/v1.5/Manifest.toml`
  [9e28174c] - BinDeps v1.0.2
  [8f4d0f93] ↑ Conda v1.4.1 ⇒ v1.5.2
  [a9693cdc] - CondaBinDeps v0.2.0
  [1b7aed40] ~ DIVAndNN v0.1.0 `https://github.com/gher-ulg/DIVAndNN.jl#master` ⇒ v0.1.0 `https://github.com/gher-ulg/DIVAndNN.jl#master`
  [f67ccb44] ↑ HDF5 v0.13.5 ⇒ v0.15.5
  [0234f1f7] ↑ HDF5_jll v1.10.5+7 ⇒ v1.12.0+1
  [23992714] ↑ MAT v0.8.1 ⇒ v0.10.1
  [739be429] ↑ MbedTLS v1.0.2 ⇒ v1.0.3
  [c8ffd9c3] ↑ MbedTLS_jll v2.16.6+1 ⇒ v2.16.8+1
  [85f8d34a] ↑ NCDatasets v0.10.4 ⇒ v0.11.5
  [7243133f] + NetCDF_jll v400.701.400+0
  [458c3c95] + OpenSSL_jll v1.1.1+6
  [69de0a69] ↑ Parsers v1.0.10 ⇒ v1.1.0
  [30578b45] - URIParser v0.4.1
  [3161d3a3] ↑ Zstd_jll v1.4.5+2

In [2]:
ARGO_ds = Dataset("simona notebooks/ARGO_Med_995_1005.nc", "r");
CORA_ds = Dataset("simona notebooks/CORA_PR_MOW_995_1005.nc", "r");
SDN_ds = Dataset("simona notebooks/SEADATANET_MOW_995_1005.nc", "r");
#EMOD_ds = Dataset("simona notebooks/EMODnetChemistry_MOW_995_1005.nc", "r");
WOD_ds = Dataset("simona notebooks/WOD_MOW_995_1005.nc", "r");
WOD_DO_ds = Dataset("simona notebooks/WOD_MOW_995_1005_DO.nc", "r");

#@show ARGO_ds;

In [20]:
@show ARGO_ds;

ARGO_ds = NCDataset: simona notebooks/ARGO_Med_995_1005.nc
Group: /

Dimensions
   index = 2016

Variables
  index   (2016)
    Datatype:    Int64
    Dimensions:  index

  TIME   (2016)
    Datatype:    Int64
    Dimensions:  index
    Attributes:
     units                = milliseconds since 2018-07-21 05:12:00
     calendar             = proleptic_gregorian

  TIME_QC   (2016)
    Datatype:    String
    Dimensions:  index

  LONGITUDE   (2016)
    Datatype:    Float64
    Dimensions:  index
    Attributes:
     _FillValue           = NaN

  LATITUDE   (2016)
    Datatype:    Float64
    Dimensions:  index
    Attributes:
     _FillValue           = NaN

  POSITION_QC   (2016)
    Datatype:    String
    Dimensions:  index

  PLATFORM_NUMBER   (2016)
    Datatype:    String
    Dimensions:  index

  CYCLE_NUMBER   (2016)
    Datatype:    Int32
    Dimensions:  index

  WMO_INST_TYPE   (2016)
    Datatype:    String
    Dimensions:  index

  DATA_TYPE   (2016)
    Datatype:    Strin

In [3]:
mask_file_woa13 = "/home/jovyan/blue-cloud-dataspace/MEI/INGV/INPUT/BATHYMETRY/gebco_2019_mask_1_8_edited_final_woa13.nc"
mask_file = "/home/jovyan/blue-cloud-dataspace/MEI/INGV/INPUT/BATHYMETRY/gebco_2019_mask_1_8_edited_final.nc"

"/home/jovyan/blue-cloud-dataspace/MEI/INGV/INPUT/BATHYMETRY/gebco_2019_mask_1_8_edited_final.nc"

In [4]:
mask_data = Dataset(mask_file, "r")

NCDataset: /home/jovyan/blue-cloud-dataspace/MEI/INGV/INPUT/BATHYMETRY/gebco_2019_mask_1_8_edited_final.nc
Group: /

Dimensions
   lon = 367
   lat = 129
   depth = 92

Variables
  lon   (367)
    Datatype:    Float32
    Dimensions:  lon
    Attributes:
     standard_name        = longitude
     long_name            = Longitude
     units                = degrees_east
     axis                 = X

  lat   (129)
    Datatype:    Float32
    Dimensions:  lat
    Attributes:
     standard_name        = latitude
     long_name            = Latitude
     units                = degrees_north
     axis                 = Y

  depth   (92)
    Datatype:    Float32
    Dimensions:  depth
    Attributes:
     standard_name        = depth
     long_name            = Depth
     units                = m
     positive             = down
     axis                 = Z

  mask   (367 × 129 × 92)
    Datatype:    Int8
    Dimensions:  lon × lat × depth
    Attributes:
     standard_name        = land_s

In [5]:
ARGO_lon = ARGO_ds["LONGITUDE"].var[:]; #X
ARGO_lat = ARGO_ds["LATITUDE"].var[:]; #Y
ARGO_depth = ARGO_ds["DEPTH"].var[:]; #Z
ARGO_time = ARGO_ds["TIME"].var[:]; #T
ARGO_temp = ARGO_ds["TEMPERATURE"].var[:]; #F
ARGO_sal = ARGO_ds["SALINITY"].var[:]; #F

CORA_lon = CORA_ds["LONGITUDE"].var[:];
CORA_lat = CORA_ds["LATITUDE"].var[:];
CORA_depth = CORA_ds["DEPTH"].var[:];
CORA_time = CORA_ds["TIME"].var[:];
CORA_temp = CORA_ds["TEMPERATURE"].var[:];
CORA_sal = CORA_ds["SALINITY"].var[:];

WOD_lon = WOD_ds["LONGITUDE"].var[:];
WOD_lat = WOD_ds["LATITUDE"].var[:];
WOD_depth = WOD_ds["DEPTH"].var[:];
WOD_time = WOD_ds["TIME"].var[:];
WOD_temp = WOD_ds["TEMPERATURE"].var[:];
WOD_sal = WOD_ds["SALINITY"].var[:];

WOD_DO_lon = WOD_DO_ds["LONGITUDE"].var[:];
WOD_DO_lat = WOD_DO_ds["LATITUDE"].var[:];
WOD_DO_depth = WOD_DO_ds["DEPTH"].var[:];
WOD_DO_time = WOD_DO_ds["TIME"].var[:];
WOD_DO_temp = WOD_DO_ds["TEMPERATURE"].var[:];
WOD_DO_sal = WOD_DO_ds["SALINITY"].var[:];

SDN_lon = SDN_ds["LONGITUDE"].var[:];
SDN_lat = SDN_ds["LATITUDE"].var[:];
SDN_depth = SDN_ds["DEPTH"].var[:];
SDN_time = SDN_ds["TIME"].var[:];
SDN_temp = SDN_ds["TEMPERATURE"].var[:];
SDN_sal = SDN_ds["SALINITY"].var[:];

In [6]:
pm,pn = DIVAnd_metric(ARGO_lon,ARGO_lat)

([7.223728853731982e-5 7.223919262451856e-5 … 7.219449624886457e-5 7.278765486636645e-5; 7.880433563776572e-5 7.880641282958606e-5 … 7.875765308671857e-5 7.940473598581975e-5; … ; 4.6090094753575274e-5 4.609130963911674e-5 … 4.6062791577479814e-5 4.6441250294734034e-5; 2.728577386639025e-5 2.7286493089278995e-5 … 2.726961014471566e-5 2.7493661052444523e-5], [0.004496601510469779 0.0002276760343086657 … 5.2746060732582096e-5 1.4623094886060076e-5; 0.004496601510469779 0.0002276760343086657 … 5.2746060732582096e-5 1.4623094886060076e-5; … ; 0.004496601510469779 0.0002276760343086657 … 5.2746060732582096e-5 1.4623094886060076e-5; 0.004496601510469779 0.0002276760343086657 … 5.2746060732582096e-5 1.4623094886060076e-5])

In [18]:
download_check(mask_data)

LoadError: UndefVarError: download_check not defined

In [27]:
bx, by, b = extract_bath(mask_file_woa13, true, -14.0:-8.0, 30.0:40.0);

LoadError: NetCDF error: Variable 'bat' not found in file /home/jovyan/blue-cloud-dataspace/MEI/INGV/INPUT/BATHYMETRY/gebco_2019_mask_1_8_edited_final_woa13.nc (NetCDF error code: -49)

In [24]:
xi,yi,mask = load_mask(mask_file_woa13,true,ARGO_lon,ARGO_lat,[1000]);


LoadError: NetCDF error: Variable 'bat' not found in file /home/jovyan/blue-cloud-dataspace/MEI/INGV/INPUT/BATHYMETRY/gebco_2019_mask_1_8_edited_final_woa13.nc (NetCDF error code: -49)

In [8]:
minimum(ARGO_lon),maximum(ARGO_lon),1/60

(-13.995999999999981, -9.27696, 0.016666666666666666)

In [9]:
len = 2.0
epsilon2 = 1 / 3;

fi, s = DIVAndrun(mask, (pm, pn), (lon_mask, lat_mask), (ARGO_lon, ARGO_lat), ARGO_temp, (len, len), epsilon2);

LoadError: UndefVarError: lon_mask not defined